# Hito 4 — Experimentos del Recomendador (Semana 11)

Notebook complementario al pipeline `src/`. Concentra la **inspección visual y comparativa** que respalda el informe LaTeX.

Secciones:
1. Datos y construcción de R.
2. TF-IDF sobre el catálogo (recomendador de contenido).
3. Comparativa de encodings y sparsity.
4. Normalizaciones y su impacto.
5. Filtrado colaborativo: ítem-ítem y ALS + lambda iteration.
6. Cold-start: comparativa cuantitativa.
7. Recomendador híbrido y ablación de pesos.

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib.pyplot as plt

REC = Path('..') / 'data' / 'recommender'
sorted(p.name for p in REC.iterdir())

## 1. Construcción de R

Definimos la unidad de sesión como **ventana de restock de 60 min** sobre eventos `IN` por household, alineado con la definición operacional de la propuesta. Para CF implícito esto da 1,177 sesiones × 50 productos con 17 % de densidad — sparsity realista para un dataset de hogar.

In [ ]:
spar = pd.DataFrame(json.load(open(REC/'sparsity_report.json')))
spar

## 2. TF-IDF sobre el catálogo

Cada producto se convierte en un documento concatenando nombre, categoría, nutriscore y buckets nutricionales. TF-IDF L2-normalizado: `cosine_sim = dot product`.

Tokens con IDF mínimo = términos ubicuos (organic, class_purchase) — el sistema los desentona automáticamente.
Tokens con IDF máximo = términos distintivos (cilantro, zucchini, category_8) — son los que diferencian un perfil de otro.

In [ ]:
X = sp.load_npz(REC/'tfidf_items.npz')
print('shape:', X.shape, 'nnz:', X.nnz)
cat = pd.read_csv(REC/'product_catalog.csv')
cat[['product_id','product_name','category','nutriscore','doc']].head()

## 3. Comparativa de encodings

El caso base **household × producto** (10 × 50) tiene densidad 100 %: todos los hogares han tocado todos los productos, por lo que el CF a ese nivel es degenerado. La unidad de **sesión** introduce sparsity y permite que el dot product entre columnas signifique co-ocurrencia genuina.

In [ ]:
ax = spar.set_index('matrix')['density'].plot.bar(figsize=(8,3), color='#1f77b4')
ax.set_ylabel('densidad'); ax.set_title('Densidad por variante de R'); plt.tight_layout()

## 4. Normalizaciones

Se aplican 5 normalizaciones sobre `R_restock_count` y se compara la distribución de valores no-cero. `tfidf_R` y `l2_row` son las que mejor preparan la matriz para ALS y para cosine sim ítem-ítem.

In [ ]:
summary = json.load(open(REC/'normalization_summary.json'))
pd.DataFrame(summary).T

## 5. Filtrado colaborativo y lambda iteration

Implementación propia de **ALS implícito** (Hu/Koren/Volinsky 2008). Se barrió λ ∈ {0.001, 0.01, 0.1, 0.5, 1, 5, 10} con `factors=16`, `alpha=20`, 8 iteraciones, hold-out 20 % de las interacciones.

In [ ]:
sweep = pd.read_csv(REC/'lambda_sweep.csv')
sweep

## 6. Cold-start

Para **sesión cold** (1 ó 2 productos seed), `partial_cf` (k-NN ítem-ítem CF) supera a popularidad pura y a content-only. Para **producto cold**, content-only es la única estrategia viable (no hay historial CF); recupera productos co-ocurrentes con precision@5 ≈ 0.80.

In [ ]:
cs1 = pd.read_csv(REC/'cold_start_session.csv')
cs2 = pd.read_csv(REC/'cold_start_session_2seed.csv')
pd.DataFrame({
    '1_seed': cs1.filter(regex='prec@5').mean(),
    '2_seed': cs2.filter(regex='prec@5').mean(),
})

## 7. Recomendador híbrido

Combinación lineal de los tres scorings, normalizados a [0, 1]. La señal `score_expiry` no mejora precision@5 sobre hold-out (no es un problema de hit), pero introduce el objetivo anti-desperdicio característico de SKI: empuja productos del inventario actual con vencimiento próximo.

In [ ]:
abl = pd.read_csv(REC/'hybrid_ablation.csv')
abl.sort_values('precision@5', ascending=False)